# Week 2 participant practical: simplices, boundary maps and homology over $\mathbb F_2$

This is the student investigation, built around the outline and filled triangle, square loop and hollow tetrahedron. The setup and mathematical framing are supplied. You must record predictions, complete short computational steps, check intermediate objects and justify an interpretation.

This practical makes the algebra visible before any TDA library is used. We will move through

$$K \longrightarrow C_p(K;\mathbb F_2) \xrightarrow{\partial_p} C_{p-1}(K;\mathbb F_2)
\longrightarrow Z_p/B_p = H_p(K;\mathbb F_2).$$

The examples are deliberately small: an outline triangle, a filled triangle, a square loop and a hollow tetrahedron.

**◇ Object check.** The simplicial complex $K$ is a collection of simplices. The chain space $C_p$ is a vector space whose chosen basis is the list of $p$-simplices. A matrix represents a boundary map only after those bases and their orders have been fixed.

**Working rule.** Run one section at a time. Before each TODO, state what shape, dimension or direction you expect in the output. Optional extensions come only after the core checkpoints agree.


**Prerequisite bridge.** Use the [algebra survival guide](../../algebra-survival-guide.qmd) if basis, kernel, image, rank or quotient are unfamiliar.


## Conventions and reading connection

The notebooks use **abstract simplicial complexes**: a simplex is represented by its set of vertices. A geometric realisation can place those simplices in Euclidean space, but the chain calculation uses only which faces belong to which simplices.

Every simplex here is non-empty, and all homology calculations use coefficients in $\mathbb F_2$. This matches the modulo-2 computational development in Edelsbrunner and Harer, Chapter IV, and complements Postol, Section 4.1. Other coefficient systems require orientation signs and may reveal torsion.

In [ ]:
from itertools import combinations
import numpy as np
import matplotlib.pyplot as plt

def simplex_key(simplex):
    return (len(simplex), tuple(simplex))

def close_under_faces(maximal_simplices):
    """Return all non-empty faces, grouped by dimension."""
    simplices = set()
    for maximal in maximal_simplices:
        maximal = tuple(sorted(maximal))
        for size in range(1, len(maximal) + 1):
            simplices.update(combinations(maximal, size))
    max_dim = max(len(s) - 1 for s in simplices)
    return {p: sorted((s for s in simplices if len(s) == p + 1))
            for p in range(max_dim + 1)}

def boundary_matrix(K, p):
    """Matrix of ∂_p over F_2; columns and rows use the displayed bases."""
    columns = K.get(p, [])
    rows = K.get(p - 1, []) if p > 0 else []
    matrix = np.zeros((len(rows), len(columns)), dtype=np.uint8)
    row_index = {simplex: i for i, simplex in enumerate(rows)}
    if p > 0:
        for j, simplex in enumerate(columns):
            for face in combinations(simplex, p):
                matrix[row_index[face], j] = 1
    return matrix

def rank_mod2(matrix):
    """Gaussian elimination over F_2."""
    A = np.array(matrix, dtype=np.uint8, copy=True) % 2
    rank = row = 0
    for col in range(A.shape[1]):
        candidates = np.flatnonzero(A[row:, col])
        if len(candidates) == 0:
            continue
        pivot = row + candidates[0]
        A[[row, pivot]] = A[[pivot, row]]
        for other in range(A.shape[0]):
            if other != row and A[other, col]:
                A[other] ^= A[row]
        rank += 1
        row += 1
        if row == A.shape[0]:
            break
    return rank

def homology_table(K):
    """Dimensions of C_p, Z_p, B_p and H_p over F_2."""
    result = []
    max_dim = max(K)
    for p in range(max_dim + 1):
        dim_C = len(K.get(p, []))
        rank_dp = rank_mod2(boundary_matrix(K, p)) if p > 0 else 0
        rank_next = rank_mod2(boundary_matrix(K, p + 1)) if p < max_dim else 0
        dim_Z = dim_C - rank_dp
        dim_B = rank_next
        result.append((p, dim_C, dim_Z, dim_B, dim_Z - dim_B))
    return result

def report(name, K):
    print(name)
    for p in sorted(K):
        print(f"  basis C_{p}: {K[p]}")
    print("  p | dim C_p | dim Z_p | dim B_p | beta_p")
    for row in homology_table(K):
        print(" ", " | ".join(map(str, row)))

def check_boundary_squared(K):
    for p in range(2, max(K) + 1):
        product = (boundary_matrix(K, p - 1) @ boundary_matrix(K, p)) % 2
        assert not product.any(), f"Boundary squared failed in dimension {p}"
    return True

outline_triangle = close_under_faces([(0, 1), (1, 2), (0, 2)])
filled_triangle = close_under_faces([(0, 1, 2)])
square_loop = close_under_faces([(0, 1), (1, 2), (2, 3), (0, 3)])
hollow_tetrahedron = close_under_faces([(0, 1, 2), (0, 1, 3),
                                        (0, 2, 3), (1, 2, 3)])

complexes = {
    "outline triangle": outline_triangle,
    "filled triangle": filled_triangle,
    "square loop": square_loop,
    "hollow tetrahedron": hollow_tetrahedron,
}
print("Defined four finite simplicial complexes over F_2.")

## 1. Observe: participant checkpoint

The first three complexes are drawn below. The hollow tetrahedron is the union of its four triangular faces. It does **not** contain the solid tetrahedral 3-simplex.

Before computing, distinguish the visible drawing from the mathematical object. In particular, the outline and filled triangles have the same vertices and edges, hence the same 1-skeleton, but they are different simplicial complexes.

In [ ]:
positions_2d = {
    "outline triangle": {0:(0,0), 1:(1,0), 2:(0.5,0.87)},
    "filled triangle": {0:(0,0), 1:(1,0), 2:(0.5,0.87)},
    "square loop": {0:(0,0), 1:(1,0), 2:(1,1), 3:(0,1)},
}
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, (name, K) in zip(axes, list(complexes.items())[:3]):
    pos = positions_2d[name]
    if K.get(2):
        for face in K[2]:
            ax.fill(*zip(*(pos[v] for v in face)), alpha=0.25, color='tab:blue')
    for edge in K.get(1, []):
        ax.plot(*zip(*(pos[v] for v in edge)), color='black')
    for v, xy in pos.items():
        ax.scatter(*xy, s=80, zorder=3)
        ax.text(xy[0], xy[1] + 0.08, str(v), ha='center')
    ax.set_title(name)
    ax.set_aspect('equal'); ax.axis('off')
plt.show()
print("The hollow tetrahedron contains four triangular faces but no 3-simplex.")

## 2. Predict: participant checkpoint

Without running the homology calculations, complete this table. Count a connected component as an $H_0$ class.

| Complex | Expected $\beta_0$ | Expected $\beta_1$ | Expected $\beta_2$ | Which simplex could fill the visible cycle? |
|---|---:|---:|---:|---|
| Outline triangle |  |  |  |  |
| Filled triangle |  |  |  |  |
| Square loop |  |  |  |  |
| Hollow tetrahedron |  |  |  |  |

Also predict whether adding the triangular 2-simplex to the outline triangle changes $C_0$, $C_1$, $C_2$, or more than one of them.

## 3. Implement: participant checkpoint

Run the setup cell. `close_under_faces` enforces downward closure. `boundary_matrix` creates the matrix of $\partial_p$ over $\mathbb F_2$: an entry is 1 when the row simplex is a codimension-one face of the column simplex.

Because $-1=1$ in $\mathbb F_2$, orientations and signs disappear. This is convenient, but it is a coefficient choice, not a property of homology in general.

### A. Read a boundary matrix

Display $\partial_1$ for the outline triangle. State which vector spaces are its domain and codomain, and explain one column in words.

In [ ]:
d1_outline = boundary_matrix(outline_triangle, 1)
print('rows, basis C_0:', outline_triangle[0])
print('columns, basis C_1:', outline_triangle[1])
print(d1_outline)

The edge cycle is the sum of all three edge-basis vectors, represented by $(1,1,1)^T$. Multiply by $\partial_1$. Is it a cycle?

In [ ]:
triangle_edge_cycle = np.ones(3, dtype=np.uint8)
# TODO: compute the boundary of triangle_edge_cycle modulo 2


### B. Check that boundaries are cycles

For the filled triangle, calculate $\partial_1\partial_2$. Then use `check_boundary_squared` on all four complexes. Explain why a zero product is the matrix form of $\partial^2=0$ and why it implies $B_1\subseteq Z_1$.

In [ ]:
d1_filled = boundary_matrix(filled_triangle, 1)
d2_filled = boundary_matrix(filled_triangle, 2)
# TODO: calculate (d1_filled @ d2_filled) % 2
# TODO: call check_boundary_squared for every complex


### C. Compute homology dimensions

For each degree,

$$\dim Z_p=\dim C_p-\operatorname{rank}\partial_p,\qquad
\dim B_p=\operatorname{rank}\partial_{p+1},$$

so $\beta_p=\dim Z_p-\dim B_p$. Use `report` for the outline triangle and square loop first. Reconcile the output with your predictions.

In [ ]:
# TODO: report the outline triangle and square loop


## 4. Compare: participant checkpoint

Keep the three vertices and three edges fixed. Change only whether the triangular 2-simplex is present. Compare the chain spaces, boundary ranks and Betti numbers of the outline and filled triangles.

In [ ]:
# TODO: report the outline triangle and filled triangle
# Which C_p and which boundary map changed?


Now inspect the hollow tetrahedron. The sum of its four triangular faces is a 2-cycle because every edge occurs twice and cancels in $\mathbb F_2$. Test that claim, then compute its homology.

In [ ]:
faces = hollow_tetrahedron[2]
hollow_surface = np.ones(len(faces), dtype=np.uint8)
d2_hollow = boundary_matrix(hollow_tetrahedron, 2)
# TODO: compute the boundary of hollow_surface and report the complex
# TODO extension: add (0,1,2,3) as a 3-simplex and predict what changes


### D. Use Euler characteristic as a cross-check

For a finite complex,

$$\chi(K)=\sum_p(-1)^p\dim C_p=\sum_p(-1)^p\beta_p.$$

Check both sides for all four examples. Explain why agreement is useful but does not prove that every individual Betti number is correct.

In [ ]:
def euler_from_simplices(K):
    return sum((-1)**p * len(K.get(p, [])) for p in K)

def euler_from_betti(K):
    return sum((-1)**p * row[-1] for p, row in enumerate(homology_table(K)))

# TODO: print both Euler-characteristic calculations for every complex.

### Quotient checkpoint

Let $z$ be the outline-triangle edge cycle. After adding the triangular face, $z=\partial_2(012)$.

1. Explain why $z\sim0$ in the filled triangle.
2. Suppose $z'$ differs from $z$ by the boundary of an available two-chain. Explain why $[z']=[z]$.
3. Complete: a homology class is one ______ class containing many cycle ______.

## 5. Interpret: participant checkpoint

Answer in complete sentences.

1. Why does the outline triangle have a nonzero $H_1$ class while the filled triangle does not, even though their vertices and edges are identical?
2. Why are the triangle edge cycle and square edge cycle representatives of classes rather than the holes themselves?
3. What fills the $H_2$ class of the hollow tetrahedron?
4. Which statements depend on using $\mathbb F_2$?
5. For an interaction network, what extra modelling claim is made when a 3-clique is filled as a 2-simplex?

**† Qualification.** These calculations describe the topology of the constructed simplicial complex. Whether that complex faithfully represents a dynamical system is a separate scientific and modelling question.